<a href="https://colab.research.google.com/github/sohailpayami2023/digital-signal-processing-python/blob/main/notebooks/00_python_foundations/03_probability_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Probability and Statistics for DSP Engineers

Noise is random. Wireless channels are random. BER is a
probability. Understanding these tools is essential for
every simulation in this series.

**Topics covered**
1. Why probability in DSP?
2. Generating random numbers
3. Histograms and probability density functions (PDF)
4. Cumulative distribution function (CDF)
5. Key distributions — Gaussian, Rayleigh, uniform
6. Sample statistics vs theoretical values
7. Central Limit Theorem
8. Monte Carlo BER simulation (BPSK)
9. MATLAB → Python quick reference


# 1. Why Probability in DSP?

| Phenomenon | Random quantity | Distribution |
|------------|----------------|-------------|
| Thermal noise | Noise voltage | Gaussian N(0,σ²) |
| Flat fading channel | Path amplitude | Rayleigh |
| Quantisation error | Rounding error | Uniform |
| Bit errors | Number of errors | Binomial |

The **Bit Error Rate (BER)** is a probability:
P(received bit ≠ transmitted bit).
We estimate it by **Monte Carlo simulation** —
transmit many bits, count how many are wrong.


# 2. Generating Random Numbers

| Function | Distribution | MATLAB equivalent |
|----------|-------------|-------------------|
| `np.random.randn(n)` | Gaussian N(0,1) | `randn(1,n)` |
| `np.random.rand(n)` | Uniform [0,1) | `rand(1,n)` |
| `np.random.randint(lo,hi,n)` | Uniform integer | `randi(hi,1,n)` |
| `np.random.rayleigh(s,n)` | Rayleigh(σ) | `raylrnd(s,1,n)` |
| `np.random.seed(k)` | Fix seed | `rng(k)` |


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.figsize' : (5, 3),
    'figure.dpi'     : 120,
    'axes.grid'      : True,
    'grid.alpha'     : 0.4,
})

# Fix seed for reproducibility — MATLAB: rng(42)
np.random.seed(42)

# Gaussian N(0,1) — MATLAB: randn(1, 5)
g = np.random.randn(5)
print('Gaussian  :', g.round(3))

# Uniform [0,1) — MATLAB: rand(1, 5)
u = np.random.rand(5)
print('Uniform   :', u.round(3))

# Scaled Gaussian: N(mu, sigma^2)
# MATLAB: mu + sigma*randn(1, 5)
mu, sigma = 5, 2
x = mu + sigma * np.random.randn(5)
print(f'N({mu},{sigma}^2):', x.round(3))

# Rayleigh — fading envelope amplitude
# MATLAB: raylrnd(1, 1, 5)
r = np.random.rayleigh(scale=1, size=5)
print('Rayleigh  :', r.round(3))

# Random integers — MATLAB: randi(3, 1, 8)
# Generates random bits or symbol indices
bits = np.random.randint(0, 2, 8)  # 8 random bits
print('Random bits:', bits)


# 3. Histograms and Probability Density Functions

A **histogram** counts how many samples fall in each bin.
Normalised to unit area (density=True), it approximates
the **Probability Density Function (PDF)**.

The PDF f(x) satisfies: P(a ≤ X ≤ b) = ∫_a^b f(x) dx

MATLAB: `histogram(x, n, "Normalization", "pdf")`


In [ ]:
np.random.seed(0)
N = 10_000

# Gaussian noise samples
x = np.random.randn(N)

fig, ax = plt.subplots(figsize=(5, 3))

# Normalised histogram ≈ PDF
# density=True: y-axis is probability density, not count
# MATLAB: histogram(x, 60, "Normalization", "pdf")
ax.hist(x, bins=60, density=True, alpha=0.6,
        color='steelblue', label='Histogram')

# Overlay theoretical Gaussian PDF
xg = np.linspace(-4, 4, 300)
# stats.norm.pdf: N(0,1) probability density
# MATLAB: normpdf(xg, 0, 1)
ax.plot(xg, stats.norm.pdf(xg), 'r', lw=2,
        label='N(0,1) PDF')

ax.set(xlabel='x', ylabel='Density',
       title='Histogram vs Theoretical PDF (N=10 000)')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Compare Gaussian and Rayleigh distributions
# Rayleigh models the envelope (magnitude) of a
# complex Gaussian channel coefficient h ~ CN(0,1)
h = (np.random.randn(N) + 1j*np.random.randn(N))
h /= np.sqrt(2)
envelope = np.abs(h)   # Rayleigh distributed

fig, axes = plt.subplots(1, 2, figsize=(5, 2.5))

axes[0].hist(np.random.randn(N), bins=60,
             density=True, alpha=0.7)
xg = np.linspace(-4, 4, 200)
axes[0].plot(xg, stats.norm.pdf(xg), 'r')
axes[0].set(xlabel='x', title='Gaussian N(0,1)')

axes[1].hist(envelope, bins=60,
             density=True, alpha=0.7, color='C1')
xr = np.linspace(0, 3, 200)
# stats.rayleigh.pdf: Rayleigh PDF
# MATLAB: raylpdf(xr, 1/sqrt(2))
axes[1].plot(xr, stats.rayleigh.pdf(xr,scale=1/np.sqrt(2)),
             'r')
axes[1].set(xlabel='|h|', title='Rayleigh (fading)')

plt.tight_layout()
plt.show()


# 4. Cumulative Distribution Function (CDF)

The **CDF** F(x) = P(X ≤ x) — the probability that a
random variable is less than or equal to x.

In DSP it answers: "What fraction of noise samples
exceed the detection threshold?" which directly gives BER.

MATLAB: `normcdf`, `raylcdf`
SciPy: `stats.norm.cdf`, `stats.rayleigh.cdf`


In [ ]:
np.random.seed(0)
x_samp = np.random.randn(N)

# Empirical CDF: sort samples, plot cumulative fraction
# MATLAB: empirical CDF via ecdf(x)
x_sorted = np.sort(x_samp)
cdf_emp  = np.arange(1, N+1) / N

xg = np.linspace(-4, 4, 300)
# Theoretical CDF — MATLAB: normcdf(xg, 0, 1)
cdf_theory = stats.norm.cdf(xg)

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(x_sorted, cdf_emp, lw=0.8, alpha=0.7,
        label='Empirical')
ax.plot(xg, cdf_theory, 'r', lw=2, label='Theoretical')
ax.set(xlabel='x', ylabel='P(X ≤ x)',
       title='CDF — Gaussian N(0,1)')
ax.legend()
plt.tight_layout()
plt.show()

# Practical use: P(noise > threshold) = 1 - CDF(threshold)
threshold = 1.0
p_exceed = 1 - stats.norm.cdf(threshold)
print(f'P(X > {threshold}) = {p_exceed:.4f}')
# MATLAB: 1 - normcdf(1, 0, 1)


# 5. Key Distributions in Wireless DSP

| Distribution | Parameters | Models | NumPy |
|-------------|-----------|--------|-------|
| Gaussian | μ, σ | Thermal noise, I/Q noise | `randn` |
| Rayleigh | σ | Fading envelope | `random.rayleigh` |
| Exponential | λ | Power in Rayleigh channel | `random.exponential` |
| Uniform | a, b | Quantisation error | `random.uniform` |
| Binomial | n, p | Number of bit errors | `random.binomial` |


In [ ]:
np.random.seed(7)
N = 5000

samples = {
    'Gaussian N(0,1)' : np.random.randn(N),
    'Rayleigh (σ=1)'  : np.random.rayleigh(1, N),
    'Uniform [0,1]'   : np.random.uniform(0, 1, N),
    'Exponential (λ=1)': np.random.exponential(1, N),
}

fig, axes = plt.subplots(1, 4, figsize=(5, 2.5))
for ax, (name, data) in zip(axes, samples.items()):
    ax.hist(data, bins=40, density=True,
            color='steelblue', alpha=0.8)
    ax.set_title(name, fontsize=7)
    ax.set_xlabel('x', fontsize=7)
    ax.tick_params(labelsize=6)
plt.tight_layout()
plt.show()


# 6. Sample Statistics vs Theoretical Values

Sample statistics (computed from data) converge to
theoretical values as the number of samples N grows.
This is the **Law of Large Numbers** — the foundation
of Monte Carlo simulation.

| Statistic | NumPy | MATLAB |
|-----------|-------|--------|
| Mean | `np.mean(x)` | `mean(x)` |
| Variance | `np.var(x)` | `var(x)` |
| Std dev | `np.std(x)` | `std(x)` |
| Median | `np.median(x)` | `median(x)` |
| Percentile | `np.percentile(x,p)` | `prctile(x,p)` |


In [ ]:
np.random.seed(0)

# Show convergence: estimate mean of N(0,1) with
# increasing number of samples
n_max   = 10_000
samples = np.random.randn(n_max)
n_vals  = np.logspace(1, 4, 50, dtype=int)

# Running mean — MATLAB: cumsum(x) ./ (1:n)
running_mean = [np.mean(samples[:n]) for n in n_vals]

fig, ax = plt.subplots(figsize=(5, 2.5))
ax.semilogx(n_vals, running_mean)
ax.axhline(0, color='r', lw=1.5, ls='--',
           label='True mean = 0')
ax.set(xlabel='Number of samples (N)',
       ylabel='Sample mean',
       title='Law of Large Numbers — convergence to true mean')
ax.legend()
plt.tight_layout()
plt.show()

# Summary statistics
x = np.random.randn(1000)
print(f'mean   : {np.mean(x):.4f}  (theory: 0)')
print(f'std    : {np.std(x):.4f}  (theory: 1)')
print(f'var    : {np.var(x):.4f}  (theory: 1)')
print(f'median : {np.median(x):.4f}')
print(f'95th % : {np.percentile(x,95):.4f}')


# 7. Central Limit Theorem

The sum of many independent random variables tends to a
Gaussian distribution, regardless of their individual
distributions. This is why thermal noise is Gaussian —
it is the sum of countless random electron motions.

It also justifies using Gaussian models for interference
in cellular networks (sum of many interferers).


In [ ]:
np.random.seed(3)
N_trials = 10_000

# Sum of k uniform [0,1] random variables
# As k increases, the sum becomes more Gaussian
k_values = [1, 2, 6, 30]

fig, axes = plt.subplots(1, 4, figsize=(5, 2.5))
xg = np.linspace(-3, 3, 200)

for ax, k in zip(axes, k_values):
    # Sum of k uniform r.v.s, then normalise to N(0,1)
    s = np.sum(np.random.rand(N_trials, k), axis=1)
    s = (s - k/2) / np.sqrt(k/12)  # standardise
    ax.hist(s, bins=50, density=True,
            alpha=0.7, color='steelblue')
    ax.plot(xg, stats.norm.pdf(xg), color='r', lw=1)
    ax.set_title(f'k={k}', fontsize=8)
    ax.tick_params(labelsize=6)
plt.suptitle('Central Limit Theorem', y=1.02, fontsize=9)
plt.tight_layout()
plt.show()


# 8. Monte Carlo BER Simulation — BPSK

Monte Carlo BER is the core technique in wireless
simulation: generate random bits, transmit through
a noisy channel, detect, count errors.

**BPSK** maps bit 0 → −1, bit 1 → +1.
The receiver decides: if received > 0, decide 1, else 0.

Theoretical BPSK BER:
**P_e = Q(√(2·E_b/N_0)) = 0.5·erfc(√(E_b/N_0))**

We compare Monte Carlo results to this formula.


In [ ]:
from scipy.special import erfc

np.random.seed(0)
n_bits   = 100_000   # bits per SNR point
snr_db   = np.arange(0, 13, 1)  # 0..12 dB

ber_mc   = np.zeros(len(snr_db))  # Monte Carlo
ber_th   = np.zeros(len(snr_db))  # Theoretical

for i, snr in enumerate(snr_db):
    snr_lin = 10 ** (snr / 10)   # Eb/N0 linear

    # Generate random bits (0 or 1)
    bits = np.random.randint(0, 2, n_bits)

    # BPSK modulation: 0 -> -1, 1 -> +1
    tx = 2*bits - 1

    # AWGN: noise std = sqrt(1 / (2*Eb/N0))
    noise_std = np.sqrt(1 / (2*snr_lin))
    rx = tx + noise_std * np.random.randn(n_bits)

    # Detect: threshold at 0
    bits_hat = (rx > 0).astype(int)

    # Count bit errors
    ber_mc[i] = np.mean(bits != bits_hat)

    # Theoretical BER
    ber_th[i] = 0.5 * erfc(np.sqrt(snr_lin))

fig, ax = plt.subplots(figsize=(5, 3))
ax.semilogy(snr_db, ber_mc, 'o-', label='Monte Carlo')
ax.semilogy(snr_db, ber_th, 'r--', label='Theoretical')
ax.set(xlabel='Eb/N0 (dB)', ylabel='BER',
       title='BPSK BER — Monte Carlo vs Theoretical')
ax.set_ylim(1e-5, 1)
ax.legend()
plt.tight_layout()
plt.show()


# 9. MATLAB → Python Quick Reference

| Operation | MATLAB | NumPy / SciPy |
|-----------|--------|---------------|
| Fix seed | `rng(42)` | `np.random.seed(42)` |
| Gaussian N(0,1) | `randn(m,n)` | `np.random.randn(m,n)` |
| Gaussian N(μ,σ²) | `mu+sigma*randn(m,n)` | `np.random.normal(mu,sigma,(m,n))` |
| Uniform [0,1] | `rand(m,n)` | `np.random.rand(m,n)` |
| Uniform [a,b] | `a+(b-a)*rand(m,n)` | `np.random.uniform(a,b,(m,n))` |
| Rayleigh | `raylrnd(s,m,n)` | `np.random.rayleigh(s,(m,n))` |
| Random integers | `randi(hi,m,n)` | `np.random.randint(0,hi,(m,n))` |
| Mean | `mean(x)` | `np.mean(x)` |
| Std dev | `std(x)` | `np.std(x)` |
| Variance | `var(x)` | `np.var(x)` |
| Median | `median(x)` | `np.median(x)` |
| Percentile | `prctile(x,p)` | `np.percentile(x,p)` |
| Histogram (count) | `hist(x,n)` | `np.histogram(x,n)` |
| Histogram (plot) | `histogram(x,n)` | `ax.hist(x,bins=n)` |
| Normalised hist | `histogram(...,"pdf")` | `ax.hist(...,density=True)` |
| Gaussian PDF | `normpdf(x,mu,s)` | `stats.norm.pdf(x,mu,s)` |
| Gaussian CDF | `normcdf(x,mu,s)` | `stats.norm.cdf(x,mu,s)` |
| Q-function | `qfunc(x)` | `0.5*erfc(x/sqrt(2))` |
| erfc | `erfc(x)` | `scipy.special.erfc(x)` |
| Sort | `sort(x)` | `np.sort(x)` |
